In [ ]:
import rawpy  # pip install rawpy
import cv2 # pip install opencv-python + pip install opencv-contrib-python
import glob
import math
import numpy as np
import os
from skimage.color import rgb2lab # pip3 install scikit-image
from PIL import Image, ImageCms
from colormath.color_objects import AdobeRGBColor, LabColor #pip3 install colormath // requires setuptools+networkx
from colormath.color_conversions import convert_color
import pandas as pd
from scipy import stats
import time
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

In [ ]:
#Segmentation in Sensor display approach images with Gaussian Mixture Model


t_inici = time.time()

def detect_chromaticity(img_bgr, colors, window_size=8, threshold=2.5):
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h, w = img_hsv.shape[:2]

    cy, cx = colors
    half = window_size // 2

    x1, x2 = max(cx - half, 0), min(cx + half + 1, w)
    y1, y2 = max(cy - half, 0), min(cy + half + 1, h)

    roi = img_hsv[y1:y2, x1:x2]

    H_roi = roi[:,:,0].flatten()
    S_roi = roi[:,:,1].flatten()
    chromaticity_samples = np.vstack([H_roi, S_roi]).T

    gmm = GaussianMixture(n_components=1, covariance_type='full', random_state=0)
    gmm.fit(chromaticity_samples)
    mean_chroma = gmm.means_[0]
    cov_chroma = gmm.covariances_[0]

    H = img_hsv[:,:,0].flatten()
    S = img_hsv[:,:,1].flatten()
    all_pixels = np.vstack([H, S]).T

    diff = all_pixels - mean_chroma
    inv_cov = np.linalg.inv(cov_chroma)
    mahalanobis_dist = np.sqrt(np.sum(diff @ inv_cov * diff, axis=1))

    mask = (mahalanobis_dist < threshold).astype(np.uint8)
    mask_img = mask.reshape(img_hsv.shape[:2])

    result = cv2.bitwise_and(img_bgr, img_bgr, mask=mask_img)

    return mask_img, result


folder = "Gamma_22"

files = [folder+"/papers_colors_v3_small_scene1_0347.png", folder+"/papers_colors_v3_small_scene2_0354.png", folder+"/papers_colors_v3_small_scene3_0364.png", folder+"/papers_colors_v3_small_scene4_0371.png"]


colors = ([420, 850], [550, 850], [690, 850], [850, 850])

df = pd.DataFrame() 
idx = 0

for f in files:
    
    if "coef" in f or "GT" in f: continue

    window_size = 4
    threshold = 4
    
    idx = 1
    
    for c in colors:
        img_bgr = cv2.imread(f)
        
        x = c[0]
        y = c[1]
            
        winsz = 10   #ajustar en f(x) del soroll/textura
    
        color_roi = (0,0,255)

    
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_bgr, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)
        
        mask, result = detect_chromaticity(img_bgr, c, window_size, threshold)
    
        # Mostrar resultats
        plt.figure(figsize=(15,5))
        plt.subplot(1,3,1)
        plt.title("Original")
        plt.imshow(cv2.cvtColor(img_roi, cv2.COLOR_BGR2RGB))
        plt.axis("off")
    
        plt.subplot(1,3,2)
        plt.title("Màscara cromaticitat")
        plt.imshow(mask, cmap="gray")
        plt.axis("off")
    
        plt.subplot(1,3,3)
        plt.title("Resultat filtrat")
        plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        plt.axis("off")
    
        plt.show()
    
        result_name = "../results/GMM_"+str(threshold)+"_"+f[-15:-4]+"_"+str(idx)+".png"
        img_save = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        cv2.imwrite (result_name, img_save)
    
        idx += 1

        t_final = time.time()
        print("time: ", t_final-t_inici)

In [ ]:
# GMM -- Precision, Recall, F1

import pandas as pd
import glob

folder = "../results"
GT_files = glob.glob("Gamma_22/*GT_2EV.png")
GT_files.sort()
results_files = glob.glob(folder+"/GMM*.png")
results_files.sort()

df = pd.DataFrame()
idx = 0

for gt in GT_files:
        img_GT = cv2.imread(gt)
        img_GT = img_GT
        for res in results_files:
            if gt[-14:-11] in res and gt[-17] == res[-5]:
                result = cv2.imread(res)
                result = result
                FN_img = cv2.subtract(img_GT, result)
                FP_img = cv2.subtract(result, img_GT)
                TP_img = cv2.bitwise_and(result, img_GT)
                #TP_img = cv2.cvtColor(TP_img, cv2.COLOR_RGB2BGR)

                FN_img_g = cv2.cvtColor(FN_img, cv2.COLOR_BGR2GRAY)
                FP_img_g = cv2.cvtColor(FP_img, cv2.COLOR_BGR2GRAY)
                TP_img_g = cv2.cvtColor(TP_img, cv2.COLOR_BGR2GRAY)
                
                size = FN_img.shape
                total = size[0]*size[1]

                FN_im = FN_img_g.flatten()
                FP_im = FP_img_g.flatten()
                TP_im = TP_img_g.flatten()

                FN_list = FN_im[FN_im>0]
                FP_list = FP_im[FP_im>0]
                TP_list = TP_im[TP_im>0]

                FN = len(FN_list)
                FP = len(FP_list)
                TP = len(TP_list)

                print ("Total: ", total)
                print ("FN: ", FN)
                print ("FP: ", FP)
                print ("TP: ", TP)
                
                TN = total - TP - FP - FN

                print ("TN: ", TN)
                
                if TP == 0: 
                    prec = 0
                    recall = 0
                    F1 = 0
                elif TP != 0:
                    #print (res)
                    prec = TP / (TP+FP)
                    #print (prec)
                    recall = TP / (TP+FN)
                    #print (recall)
                    F1 = 2*(prec*recall)/(prec + recall)
                                
                #print ("Image: ", gt[58],    "Paper: ", gt[60:63])
                #print ("Classifier: ", clas)
                #print ("Precision: ", prec)
                #print ("Recall: ", recall)
                #print ("F1: ", F1)
                
                if "p1" in gt:
                    paper = 1
                elif "p2" in gt:
                    paper = 2
                elif "p3" in gt:
                    paper = 3
                elif "p4" in gt:
                    paper = 4
                
                df.at[idx, "File"] = res
                df.at[idx, "Image"] = gt[-23]
                df.at[idx, "Paper"] = paper
                df.at[idx, "Method"] = "GMM"
                df.at[idx, "TP"] = TP
                df.at[idx, "FP"] = FP
                df.at[idx, "FN"] = FN
                df.at[idx, "Precision"] = prec
                df.at[idx, "Recall"] = recall
                df.at[idx, "F1"] = F1
                
                idx += 1

print(df)
df.to_excel("../results/GMM_F1.xlsx") 